In [7]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime, timedelta
import pytz
from pathlib import Path
import time

# ===================== CONFIG =====================
PAIRS = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF']

# Timeframes corrigés (noms valides pour les fichiers)
TIMEFRAMES = {
    'M1':  mt5.TIMEFRAME_M1,      # 1 minute
    'M3':  mt5.TIMEFRAME_M3,      # 3 minutes
    'H1':  mt5.TIMEFRAME_H1,      # 1 heure
    'H4':  mt5.TIMEFRAME_H4,      # 4 heures
    'D1':  mt5.TIMEFRAME_D1,      # 1 jour
    'W1':  mt5.TIMEFRAME_W1,      # 1 semaine
    'MN1': mt5.TIMEFRAME_MN1      # 1 mois
}

CLEAN_DIR = Path("clean_multi_tf")
CLEAN_DIR.mkdir(exist_ok=True)

MT5_PATH = r"C:\Program Files\MetaTrader 5\terminal64.exe"   # ← CHANGE CE CHEMIN !

RETURN_CAP_PERCENTILE = 0.005
RANGE_CAP_PERCENTILE = 0.005

FORCE_FULL_RELOAD = True   # Mets False après le premier chargement complet

# ===================== CONNEXION =====================
def connect_mt5():
    if not mt5.initialize(path=MT5_PATH):
        print("❌ Erreur MT5:", mt5.last_error())
        return False
    print("✅ Connecté à MetaTrader 5")
    return True

# ===================== NETTOYAGE =====================
def clean_df(df: pd.DataFrame, pair: str, tf: str):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['time'], unit='s')
    
    df['pair'] = pair
    df['timeframe'] = tf
    df['source'] = 'mt5_multi_tf'

    # Calculs de nettoyage
    df['return'] = df['close'].pct_change()
    df['range'] = (df['high'] - df['low']) / df['close']

    # Clipping des extrêmes
    lower_r = df['return'].quantile(RETURN_CAP_PERCENTILE)
    upper_r = df['return'].quantile(1 - RETURN_CAP_PERCENTILE)
    df['return'] = df['return'].clip(lower_r, upper_r)

    lower_range = df['range'].quantile(RANGE_CAP_PERCENTILE)
    upper_range = df['range'].quantile(1 - RANGE_CAP_PERCENTILE)
    df['range'] = df['range'].clip(lower_range, upper_range)

    # Validation OHLC
    df['high'] = df[['high', 'close', 'low']].max(axis=1)
    df['low']  = df[['high', 'close', 'low']].min(axis=1)

    # Colonnes finales
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'tick_volume',
             'return', 'range', 'pair', 'timeframe', 'source']]
    df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    
    df = df.drop_duplicates(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    return df

# ===================== CHARGEMENT =====================
def load_or_update_pair_tf(pair, tf_name, tf_code):
    file_path = CLEAN_DIR / f"{pair}_{tf_name}_clean.csv"
    
    if FORCE_FULL_RELOAD or not file_path.exists():
        from_date = datetime(2016, 1, 1, tzinfo=pytz.UTC)
        print(f"🆕 {pair} {tf_name} → Chargement complet depuis 2016")
    else:
        df_old = pd.read_csv(file_path, parse_dates=['timestamp'])
        last_date = df_old['timestamp'].max()
        from_date = last_date + timedelta(minutes=1)
        print(f"📌 {pair} {tf_name} → Mise à jour depuis {from_date.date()}")

    to_date = datetime.now(pytz.UTC) + timedelta(days=1)

    rates = mt5.copy_rates_range(pair, tf_code, from_date, to_date)
    
    if rates is None or len(rates) == 0:
        print(f"⚠️ Aucune donnée pour {pair} {tf_name}")
        return None

    df_new = pd.DataFrame(rates)
    df_clean = clean_df(df_new, pair, tf_name)

    try:
        df_clean.to_csv(file_path, index=False)
        print(f"✅ {pair} {tf_name} → {len(df_clean):,} lignes sauvegardées\n")
    except Exception as e:
        print(f"❌ Erreur sauvegarde {pair} {tf_name} : {e}\n")
    
    return df_clean

# ===================== MAIN =====================
if __name__ == "__main__":
    print("="*80)
    print("🚀 MULTI-TIMEFRAME FX DATA LOADER")
    print("   Chargement depuis 2016 + Nettoyage")
    print("="*80, "\n")

    if not connect_mt5():
        exit()

    for pair in PAIRS:
        print(f"\n=== TRAITEMENT DE LA PAIRE : {pair} ===")
        for tf_name, tf_code in TIMEFRAMES.items():
            load_or_update_pair_tf(pair, tf_name, tf_code)

    print("\n🎉 TOUS LES TIMEFRAMES ONT ÉTÉ TRAITÉS AVEC SUCCÈS !")
    print(f"📁 Dossier : {CLEAN_DIR.resolve()}\n")

    # Résumé final
    print("Résumé des fichiers créés :")
    for file in sorted(CLEAN_DIR.glob("*.csv")):
        try:
            size = len(pd.read_csv(file))
            print(f"   • {file.name:<25} → {size:,} lignes")
        except:
            print(f"   • {file.name:<25} → Erreur lecture")

    mt5.shutdown()

🚀 MULTI-TIMEFRAME FX DATA LOADER
   Chargement depuis 2016 + Nettoyage

✅ Connecté à MetaTrader 5

=== TRAITEMENT DE LA PAIRE : EURUSD ===
🆕 EURUSD M1 → Chargement complet depuis 2016
✅ EURUSD M1 → 3,805,376 lignes sauvegardées

🆕 EURUSD M3 → Chargement complet depuis 2016
✅ EURUSD M3 → 1,269,698 lignes sauvegardées

🆕 EURUSD H1 → Chargement complet depuis 2016
✅ EURUSD H1 → 63,557 lignes sauvegardées

🆕 EURUSD H4 → Chargement complet depuis 2016
✅ EURUSD H4 → 15,913 lignes sauvegardées

🆕 EURUSD D1 → Chargement complet depuis 2016
✅ EURUSD D1 → 2,656 lignes sauvegardées

🆕 EURUSD W1 → Chargement complet depuis 2016
✅ EURUSD W1 → 534 lignes sauvegardées

🆕 EURUSD MN1 → Chargement complet depuis 2016
✅ EURUSD MN1 → 123 lignes sauvegardées


=== TRAITEMENT DE LA PAIRE : GBPUSD ===
🆕 GBPUSD M1 → Chargement complet depuis 2016
✅ GBPUSD M1 → 3,805,091 lignes sauvegardées

🆕 GBPUSD M3 → Chargement complet depuis 2016
✅ GBPUSD M3 → 1,269,792 lignes sauvegardées

🆕 GBPUSD H1 → Chargement compl